# Step 0.3 - Bayer DINO image embeddings of Cell Painting wells

**What this notebook does.** It attaches compound information (identifiers, SMILES, InChIKey) to image
embeddings of JUMP Cell Painting wells produced by Bayer with a DINO model, and saves the result as Parquet.

**Background.** In a *Cell Painting* experiment, cells are treated with a compound, stained with dyes
that highlight different parts of the cell (nucleus, mitochondria, cytoskeleton, …) and photographed
under a microscope. The images record how the compound changes the cells' appearance, their
*morphology*. **DINO** is a vision-transformer model trained with *self-supervised learning*: it learns
to describe images without any human labels. Kim *et al.* (Bayer) trained DINO on images from the
[JUMP Cell Painting](https://jump-cellpainting.broadinstitute.org/) dataset and released one embedding
vector per well for a subset of compounds.

Two versions of the embeddings are processed:
* **Raw** ("no post-processing") - embeddings exactly as produced by the model.
* **Normalized** ("sphering + MAD robustize") - the same embeddings after the authors corrected for
  plate-to-plate technical variation using DMSO control wells (these two operations are explained in
  Step 1.1).

**Inputs.** Bayer well-level feature files (`..._well_features.csv.gz`) and the JUMP compound metadata table.

**Outputs.** `BAYER_DINO_embeddings_nonorm_final.parquet` (raw) and
`BAYER_DINO_embeddings_normed_final.parquet` (normalized). Each row is one **well**; a compound
usually has several wells (replicates and/or doses).

**Where to run it.** Written for Google Colab / Colab Enterprise: it uses the `gsutil` command-line tool and
caches immutable downloads under the repository-level `data/raw/` directory and retains assembled local
outputs under this stage's `results/` directory. You need read/write access to the
`gs://calico-reference-embeddings` Cloud Storage bucket for the remote sources and uploads.

## Setup

All libraries used in this notebook are imported in the next cell. Locally they are installed by `uv sync`
(see the main README); on Google Colab, first run
`!pip install git+https://github.com/axiomcura/ModelBenchmarking.git`, which installs the same dependencies.

In [1]:
import os
import time

import duckdb
import pandas as pd

from model_benchmarking.paths import cache_gcs_file, notebook_artifact_dirs

RAW_DATA_DIR, RESULTS_DIR = notebook_artifact_dirs("0.download_and_assemble", "bayer_dino")
METADATA_RAW_DIR, _ = notebook_artifact_dirs("0.download_and_assemble", "compound_metadata")


## Part A - Raw embeddings

### A.1 Configuration

In [2]:
gcs_bayer = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/source_csvs/Bayer_DINO_embeddings__Kim_et_al_subset__no_postprocessing_well_features.csv.gz'
gcs_meta = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'

gcs_output = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/BAYER_DINO_embeddings_nonorm_final.parquet'

# Local staging paths
local_bayer = RAW_DATA_DIR / 'bayer_nonorm.csv.gz'
local_meta = METADATA_RAW_DIR / 'JUMP_CP_compound_metadata_with_SMILES_standard.csv'
local_output = RESULTS_DIR / 'BAYER_DINO_embeddings_nonorm_final.parquet'

### A.2 Copy the files to local disk

In [3]:
print("📥 Downloading files to local SSD for fast DuckDB processing...")
if not os.path.exists(local_bayer):
    os.system(f"gsutil cp {gcs_bayer} {local_bayer}")
if not os.path.exists(local_meta):
    os.system(f"gsutil cp {gcs_meta} {local_meta}")
print("   ✅ Files staged locally.")

### A.3 Find the shared identifier column
Only the headers are read. The metadata columns that also exist in the embedding file are excluded from the join result, so no column is duplicated.

In [4]:
print("\n🔍 Inspecting CSV headers...")
con = duckdb.connect()

# Read just the headers of both files (takes a fraction of a second)
bayer_cols = con.execute(f"DESCRIBE SELECT * FROM '{local_bayer}'").df()['column_name'].tolist()
meta_cols = con.execute(f"DESCRIBE SELECT * FROM '{local_meta}'").df()['column_name'].tolist()

possible_join_cols = ['Metadata_JCP2022', 'perturbation_ID', 'perturbation_id', 'SMILES_standard', 'ID']
JOIN_COL = next((c for c in possible_join_cols if c in bayer_cols and c in meta_cols), None)

if not JOIN_COL:
    raise KeyError(f"❌ Could not find a shared identifier column to join on! \nBayer: {bayer_cols[:5]}\nMeta: {meta_cols[:5]}")

print(f"   -> Found shared identifier: '{JOIN_COL}'")

# Find shared columns to exclude from metadata (so we don't get duplicate columns)
exclude_meta_cols = [c for c in meta_cols if c in bayer_cols and c != JOIN_COL]

# Build the dynamic EXCLUDE string for the SQL query
cols_to_exclude = ['rn', JOIN_COL] + exclude_meta_cols
exclude_str = "EXCLUDE (" + ", ".join([f'"{c}"' for c in cols_to_exclude]) + ")"

### A.4 Join embeddings and metadata
The metadata is reduced to one row per identifier, so the join cannot duplicate wells, and a **left join** keeps every well, including DMSO control wells without compound metadata. `ignore_errors=true` lets DuckDB skip malformed CSV lines.

> ⚠️ Lines skipped by `ignore_errors=true` are neither counted nor reported. Compare the row count printed in the next step with the number of rows in the source file.

In [5]:
print("\n⚙️ Executing out-of-core LEFT JOIN directly to Parquet...")
start_time = time.time()

# 🚨 THE FIX: Added sample_size=-1, null_padding=true, and ignore_errors=true
# to the Bayer read_csv_auto function so it powers through bad lines!

query = f"""
CREATE OR REPLACE TABLE clean_meta AS
SELECT *
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY "{JOIN_COL}") as rn
    FROM read_csv_auto('{local_meta}', all_varchar=true)
)
WHERE rn = 1;

COPY (
    SELECT
        b.*,
        m.* {exclude_str}
    FROM read_csv_auto('{local_bayer}', sample_size=-1, null_padding=true, ignore_errors=true) AS b
    LEFT JOIN clean_meta AS m
    ON b."{JOIN_COL}" = m."{JOIN_COL}"
) TO '{local_output}' (FORMAT PARQUET, COMPRESSION 'snappy');
"""

try:
    con.execute(query)
    print(f"   ✅ DuckDB assembly complete in {time.time() - start_time:.1f} seconds!")
except Exception as e:
    raise RuntimeError(f"DuckDB Merge Error: {e}")
finally:
    con.close()

print(f"   ✅ DuckDB assembly complete in {time.time() - start_time:.1f} seconds!")

### A.5 Upload and check the result

In [6]:
print(f"\n📤 Uploading fully annotated Parquet to Google Cloud Storage...")
os.system(f"gsutil cp {local_output} {gcs_output}")

# Verify the row count matches exactly!
df_final = pd.read_parquet(local_output)
print("-" * 60)
print(f"📊 Final Parquet Shape: {df_final.shape[0]:,} Rows, {df_final.shape[1]:,} Columns")
print("-" * 60)

# Optional cleanup to free SSD space
print(f"Raw downloads retained under: {RAW_DATA_DIR.parent}")
print(f"Local result retained at: {local_output}")

print(f"🎉 SUCCESS! File saved permanently to: {gcs_output}")

## Part B - Normalized embeddings, first version (kept for the record)

An earlier way of preparing the normalized embeddings. It differs from Part C in two ways: it keeps
**one well per compound** (later duplicates of an InChIKey are dropped) and it uses an **inner** join.
It writes `BAYER_DINO_embeddings_norm_final.parquet` (*norm*, not *normed*), which is **not** used by
later steps.

### B.1 Configuration

In [7]:
# Normalized Bayer source and shared compound metadata
bayer_gcs = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/source_csvs/Bayer_DINO_embeddings__Kim_et_al_subset__sphering_mad_robustize_well_features.csv.gz'
metadata_gcs = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'
bayer_csv = cache_gcs_file(bayer_gcs, RAW_DATA_DIR / 'bayer_normed.csv.gz')
metadata_csv = cache_gcs_file(metadata_gcs, METADATA_RAW_DIR / 'JUMP_CP_compound_metadata_with_SMILES_standard.csv')

# Retain a local result and upload the same table to GCS.
output_parquet = RESULTS_DIR / 'BAYER_DINO_embeddings_norm_final.parquet'
gcs_output_parquet = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/BAYER_DINO_embeddings_norm_final.parquet'


### B.2 Load the embeddings and the metadata

In [8]:
print("📂 Loading Bayer DINO Embeddings (Decompressing GZIP)...")
df_bayer = pd.read_csv(bayer_csv, compression='gzip')
print(f"   ✅ Loaded {len(df_bayer):,} rows and {df_bayer.shape[1]:,} columns.")

print("📂 Loading Metadata...")
df_meta = pd.read_csv(metadata_csv)
print(f"   ✅ Loaded {len(df_meta):,} rows.")

### B.3 Find the shared identifier column

In [9]:
print("\n🔍 Auto-detecting shared identifier columns...")

# Check for common JUMP-CP IDs to use as the bridge
possible_join_cols = ['Metadata_JCP2022', 'perturbation_ID', 'perturbation_id', 'SMILES_standard', 'ID']

JOIN_COL = next((col for col in possible_join_cols if col in df_bayer.columns and col in df_meta.columns), None)

if not JOIN_COL:
    print(f"\n❌ Error: Could not automatically find a matching ID column between the two files.")
    print(f"Bayer Columns:    {list(df_bayer.columns[:10])}")
    print(f"Metadata Columns: {list(df_meta.columns[:10])}")
    raise KeyError("Please manually check the output above to see what the shared ID column is named.")

print(f"   -> Found shared identifier for joining: '{JOIN_COL}'")

# Auto-detect the exact spelling of InChIKey (InChIKey vs Metadata_InChIKey)
INDEX_COL = next((c for c in df_meta.columns if 'inchikey' in c.lower()), None)
if not INDEX_COL:
    raise KeyError("❌ Could not find an InChIKey column in the Metadata CSV!")
print(f"🔑 Target index column identified as: '{INDEX_COL}'")

### B.4 Join embeddings and metadata (inner join)

In [10]:
print("\n⚙️ Joining metadata onto the Bayer embeddings...")

# Deduplicate metadata on the join column to avoid accidental row duplication
df_meta = df_meta.drop_duplicates(subset=[JOIN_COL])

# Drop any shared columns (except the join column) to prevent messy _x and _y duplicate columns
shared_cols = [c for c in df_meta.columns if c in df_bayer.columns and c != JOIN_COL]
if shared_cols:
    df_meta = df_meta.drop(columns=shared_cols)

# Inner join so we only keep molecules that have both embeddings and metadata
df_joined = pd.merge(df_bayer, df_meta, on=JOIN_COL, how='inner')
print(f"   ✅ Join complete! Shape: {df_joined.shape[0]:,} Rows, {df_joined.shape[1]:,} Columns")

### B.5 Index by InChIKey (one well per compound)

In [11]:
print(f"\n🧹 Cleaning data types and setting '{INDEX_COL}' as the primary index...")

# Drop rows where InChIKey is missing
df_joined = df_joined.dropna(subset=[INDEX_COL])
df_joined[INDEX_COL] = df_joined[INDEX_COL].astype(str).str.strip()

# Deduplicate by InChIKey (keeping the first instance) to make the index mathematically unique
df_joined = df_joined[~df_joined[INDEX_COL].duplicated(keep='first')]

# Set InChIKey as the primary index
df_joined.set_index(INDEX_COL, inplace=True)

# Rearrange columns to put the metadata text on the left, followed by the float dimensions
metadata_cols = df_joined.select_dtypes(exclude=['number']).columns.tolist()
embedding_cols = [c for c in df_joined.columns if c not in metadata_cols]
df_joined = df_joined[metadata_cols + embedding_cols]

# Prevent PyArrow Parquet crashes by forcing all metadata columns (like 'plate') to be explicitly strings
for col in metadata_cols:
    df_joined[col] = df_joined[col].astype(str)

### B.6 Save

In [12]:
print("\n💾 Saving compressed Parquet locally and to Google Cloud Storage...")
df_joined.to_parquet(output_parquet, compression='snappy')
df_joined.to_parquet(gcs_output_parquet, compression='snappy')

print("🎉 SUCCESS! Files saved to:")
print(f"   Local: {output_parquet}")
print(f"   GCS:   {gcs_output_parquet}")

# Display the first 3 rows to verify the index, metadata, and expanded columns
print("\n--- PREVIEW OF ASSEMBLED PARQUET ---")
display(df_joined.head(3))

## Part C - Normalized embeddings (used by later steps)

The same procedure as Part A, applied to the normalized CSV. It produces
`BAYER_DINO_embeddings_normed_final.parquet`, which Steps 1–4 use.

> **Change from the original notebook.** Parts A and C originally used the same local file names and
> only downloaded a file if it was not already on disk. If Part A stopped before deleting its files,
> Part C would have silently reused the **raw** CSV and saved it as "normalized". The local file names
> in C.1 are now distinct from Part A's.

### C.1 Configuration

In [13]:
gcs_bayer = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/source_csvs/Bayer_DINO_embeddings__Kim_et_al_subset__sphering_mad_robustize_well_features.csv.gz'
gcs_meta = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv'

gcs_output = 'gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/Bayer_DINO_embeddings/BAYER_DINO_embeddings_normed_final.parquet'

# Local staging paths
# Distinct from Part A's names, so a leftover raw file can never be reused here
local_bayer = RAW_DATA_DIR / 'bayer_normed.csv.gz'
local_meta = METADATA_RAW_DIR / 'JUMP_CP_compound_metadata_with_SMILES_standard.csv'
local_output = RESULTS_DIR / 'BAYER_DINO_embeddings_normed_final.parquet'

### C.2 Copy the files to local disk

In [14]:
print("📥 Downloading files to local SSD for fast DuckDB processing...")
if not os.path.exists(local_bayer):
    os.system(f"gsutil cp {gcs_bayer} {local_bayer}")
if not os.path.exists(local_meta):
    os.system(f"gsutil cp {gcs_meta} {local_meta}")
print("   ✅ Files staged locally.")

### C.3 Find the shared identifier column
Only the headers are read. The metadata columns that also exist in the embedding file are excluded from the join result, so no column is duplicated.

In [15]:
print("\n🔍 Inspecting CSV headers...")
con = duckdb.connect()

# Read just the headers of both files (takes a fraction of a second)
bayer_cols = con.execute(f"DESCRIBE SELECT * FROM '{local_bayer}'").df()['column_name'].tolist()
meta_cols = con.execute(f"DESCRIBE SELECT * FROM '{local_meta}'").df()['column_name'].tolist()

possible_join_cols = ['Metadata_JCP2022', 'perturbation_ID', 'perturbation_id', 'SMILES_standard', 'ID']
JOIN_COL = next((c for c in possible_join_cols if c in bayer_cols and c in meta_cols), None)

if not JOIN_COL:
    raise KeyError(f"❌ Could not find a shared identifier column to join on! \nBayer: {bayer_cols[:5]}\nMeta: {meta_cols[:5]}")

print(f"   -> Found shared identifier: '{JOIN_COL}'")

# Find shared columns to exclude from metadata (so we don't get duplicate columns)
exclude_meta_cols = [c for c in meta_cols if c in bayer_cols and c != JOIN_COL]

# Build the dynamic EXCLUDE string for the SQL query
cols_to_exclude = ['rn', JOIN_COL] + exclude_meta_cols
exclude_str = "EXCLUDE (" + ", ".join([f'"{c}"' for c in cols_to_exclude]) + ")"

### C.4 Join embeddings and metadata
The metadata is reduced to one row per identifier, so the join cannot duplicate wells, and a **left join** keeps every well, including DMSO control wells without compound metadata. `ignore_errors=true` lets DuckDB skip malformed CSV lines.

> ⚠️ Lines skipped by `ignore_errors=true` are neither counted nor reported. Compare the row count printed in the next step with the number of rows in the source file.

In [16]:
print("\n⚙️ Executing out-of-core LEFT JOIN directly to Parquet...")
start_time = time.time()

# 🚨 THE FIX: Added sample_size=-1, null_padding=true, and ignore_errors=true
# to the Bayer read_csv_auto function so it powers through bad lines!

query = f"""
CREATE OR REPLACE TABLE clean_meta AS
SELECT *
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY "{JOIN_COL}") as rn
    FROM read_csv_auto('{local_meta}', all_varchar=true)
)
WHERE rn = 1;

COPY (
    SELECT
        b.*,
        m.* {exclude_str}
    FROM read_csv_auto('{local_bayer}', sample_size=-1, null_padding=true, ignore_errors=true) AS b
    LEFT JOIN clean_meta AS m
    ON b."{JOIN_COL}" = m."{JOIN_COL}"
) TO '{local_output}' (FORMAT PARQUET, COMPRESSION 'snappy');
"""

try:
    con.execute(query)
    print(f"   ✅ DuckDB assembly complete in {time.time() - start_time:.1f} seconds!")
except Exception as e:
    raise RuntimeError(f"DuckDB Merge Error: {e}")
finally:
    con.close()

print(f"   ✅ DuckDB assembly complete in {time.time() - start_time:.1f} seconds!")

### C.5 Upload and check the result

In [17]:
print(f"\n📤 Uploading fully annotated Parquet to Google Cloud Storage...")
os.system(f"gsutil cp {local_output} {gcs_output}")

# Verify the row count matches exactly!
df_final = pd.read_parquet(local_output)
print("-" * 60)
print(f"📊 Final Parquet Shape: {df_final.shape[0]:,} Rows, {df_final.shape[1]:,} Columns")
print("-" * 60)

# Optional cleanup to free SSD space
print(f"Raw downloads retained under: {RAW_DATA_DIR.parent}")
print(f"Local result retained at: {local_output}")

print(f"🎉 SUCCESS! File saved permanently to: {gcs_output}")